In [ ]:
# in this notebook we'll introduce and test the CK sampling model

In [ ]:
import sys
# Add the local stonesoup directory to sys.path
project_path = r"C:\Users\joesb\Documents\stonesoup"  # Adjust this to your actual path
if project_path not in sys.path:
    sys.path.insert(0, project_path)

#Imports
# --- Standard library ---
import sys
from datetime import datetime, timedelta

# --- Third-party libraries ---
import numpy as np
from scipy.stats import multivariate_normal, invgamma
import plotly.graph_objects as go
from types import FunctionType

# --- Stone Soup base and utilities ---
from stonesoup.base import Property
from stonesoup.types.numeric import Probability
from stonesoup.types.array import StateVector, StateVectors
from stonesoup.types.state import (State, CategoricalState, GaussianState, MarginalisedParticleState)
from stonesoup.types.detection import Detection
from stonesoup.types.track import Track
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState

# --- Stone Soup transition and driver models ---
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel, ConstantVelocity
)
from stonesoup.models.transition.categorical import MarkovianTransitionModel
from stonesoup.models.transition.levy_linear import (
    LevyLangevin, CombinedLinearLevyTransitionModel
)
from stonesoup.models.driver import AlphaStableNSMDriver
from stonesoup.models.base_driver import NoiseCase

# --- Stone Soup measurement models ---
from stonesoup.models.measurement.linear import LinearGaussian

# --- Stone Soup prediction/updating ---
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.updater.kalman import KalmanUpdater
from stonesoup.updater.particle import MarginalisedParticleUpdater

# --- Stone Soup resampling ---
from stonesoup.resampler.particle import SystematicResampler

# --- Stone Soup smoothing ---
from stonesoup.smoother.kalman import KalmanSmoother
from stonesoup.smoother.particle import (
    MarginalisedKalmanSmoother, ParticleSmoother, CarterKohnSampler
)

# --- Stone Soup plotting ---
from stonesoup.plotter import Plotterly, AnimatedPlotterly, Dimension, MetricPlotter

# --- Stone Soup metrics ---
from stonesoup.metricgenerator.ospametric import OSPAMetric
from stonesoup.metricgenerator.tracktotruthmetrics import SIAPMetrics
from stonesoup.metricgenerator.uncertaintymetric import SumofCovarianceNormsMetric
from stonesoup.metricgenerator.manager import MultiManager
from stonesoup.dataassociator.tracktotrack import TrackToTruth
from stonesoup.measures import Euclidean

start_time = datetime.now().replace(microsecond=0)

# Step 0: Define the model inputs for the specific 2-model thing.

In [ ]:
def conditional_models_funct(parameters,C):
    """
    Factory function to create a transition model with parameters.
    """
    #In this model, indicator K(t)=1 means the models have variance multiplied by C at time t.
    # Unpack the parameters
    sigma2, tau2 = parameters

    # Define the transition model with the parameters
    q=np.sqrt(tau2)
    transition_model0 = CombinedLinearGaussianTransitionModel([ConstantVelocity(q),
                                                          ConstantVelocity(q)])
    
    q=np.sqrt(C*tau2)
    transition_model1 = CombinedLinearGaussianTransitionModel([ConstantVelocity(q),
                                                          ConstantVelocity(q)])
    
    noise_scale=sigma2
    measurement_model0 = LinearGaussian(
                ndim_state=4,
                mapping=(0, 2),  
                noise_covar=np.array([[noise_scale, 0],  
                                    [0, noise_scale]])
                )
    
    noise_scale=C*sigma2
    measurement_model1 = LinearGaussian(
                ndim_state=4,  
                mapping=(0, 2),  
                noise_covar=np.array([[noise_scale, 0],
                                    [0, noise_scale]])
                )
    models={"transition":[transition_model0,transition_model1],
    "measurement":[measurement_model0,measurement_model1]}
    return models

print(type(conditional_models_funct))
def sample_sigma2_posterior(observations,generated_states,likelihood_term):
    """
    Sample sigma2 from its posterior distribution (Inverse Gamma).
    
    Args:
        observations (list): List of observations.
        states (list): List of states.
    
    Returns:
        float: Sampled value of sigma2.
    """
    
    # Compute residuals
    # TODO: For each t, pick whichever measurement model is indicated by K[t]. 
    # Then compute e(t) = y(t) - H_k(t)* x(t).

    Beta_sigma = 1e-6  # small prior "scale"

    # Number of scalar residuals is n_measurements * measurement_dimension
    n_measurements = len(observations)
    dim_measurements = (observations[0].state_vector).shape[0]  # because x- and y- measurements
    alpha_post = (dim_measurements * n_measurements)/2.0      # plus any prior shape
    beta_post  = 0.5*likelihood_term + Beta_sigma
    sigma2_sample = invgamma.rvs(a=alpha_post, scale=beta_post)

    # Sample from the posterior
    return sigma2_sample

def sample_tau2_posterior(observations, generated_states, likelihood_term):
    """
    Sample tau2 from its posterior distribution (Gamma).
    
    Args:
        states (list): List of states.
    
    Returns:
        float: Sampled value of tau2.
    """
    n_states = len(generated_states)

    # For tau^2:
    # Because states is length n, but increments are among n-1 steps:
    dim_state = (generated_states[0].state_vector).shape[0]
    alpha_post = 0.5 * (dim_state*(n_states-1)) - 2  # minus 2 if your derivation says so
    beta_post = 0.5*likelihood_term
    tau2_sample = invgamma.rvs(a=alpha_post, scale=beta_post)

    return tau2_sample

In [ ]:
def plot_sigma_tau(parameters, indicator_variables):
    # Extract sigma2 and tau2 values from the parameters list
    K_t_list=[int(np.argmax(state.state_vector)) for state in indicator_variables]
    sigma2_values = [params[0] for params,K_t in zip(parameters,K_t_list)]
    tau2_values = [params[1] for params,K_t in zip(parameters,K_t_list)]

    # Number of iterations
    n_iterations = len(parameters)

    # Create the plot for sigma
    fig_sigma = go.Figure()

    # Add the sigma values trace
    fig_sigma.add_trace(
        go.Scatter(
            x=list(range(n_iterations)),
            y=sigma2_values,
            mode='lines',
            name='Estimated σ^2',
            line=dict(color='blue', width=2)
        )
    )

    # Update layout for sigma plot
    fig_sigma.update_layout(
        plot_bgcolor="white",
        xaxis=dict(
            showgrid=True,
            gridcolor="gray",
            title=dict(text="Iteration", font=dict(size=20)),
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor="gray",
            title=dict(text="σ^2", font=dict(size=20)),
        ),
        legend=dict(
            font=dict(size=18),
            orientation='v',
            xanchor="auto",
            yanchor="auto",
            bordercolor="Black",
            borderwidth=2,
        ),
        title=dict(text="Convergence of σ^2", font=dict(size=24)),
    )

    # Create the plot for tau
    fig_tau = go.Figure()

    # Add the tau values trace
    fig_tau.add_trace(
        go.Scatter(
            x=list(range(n_iterations)),
            y=tau2_values,
            mode='lines',
            name='Estimated τ^2',
            line=dict(color='green', width=2)
        )
    )

    # Update layout for tau plot
    fig_tau.update_layout(
        plot_bgcolor="white",
        xaxis=dict(
            showgrid=True,
            gridcolor="gray",
            title=dict(text="Iteration", font=dict(size=20)),
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor="gray",
            title=dict(text="τ^2", font=dict(size=20)),
        ),
        legend=dict(
            font=dict(size=18),
            orientation='v',
            xanchor="auto",
            yanchor="auto",
            bordercolor="Black",
            borderwidth=2,
        ),
        title=dict(text="Convergence of τ^2", font=dict(size=24)),
    )


    fig_sigma.update_yaxes(type="log")
    fig_tau.update_yaxes(type="log")

    # Show the plots
    fig_sigma.show()
    fig_tau.show()

    # Create the plot for sigma
    fig_K = go.Figure()

    # Add the sigma values trace
    fig_K.add_trace(
        go.Scatter(
            x=list(range(n_iterations)),
            y=K_t_list,
            mode='lines',
            name='indicator values',
            line=dict(color='blue', width=2)
        )
    )

    fig_K.show()

In [ ]:
def plotOSPA_filtered_CK_RTS(truth,track,generated_states,KS_path):
    tracking_filters = ["unsmoothed", 
                        "CK-smoothed", 
                        "RTS_smoothed",            
                        ]



    ospa_generators = [OSPAMetric(c=40, p=1,
                                generator_name=f'{tracking_filter} OSPA metrics',
                                tracks_key=f'tracks_{tracking_filter}',
                                truths_key='truths'
                                )
                    for tracking_filter in tracking_filters]


    siap_generators = [SIAPMetrics(position_measure=Euclidean((0, 2)),
                                velocity_measure=Euclidean((1, 3)),
                                generator_name=f'{tracking_filter} SIAP metrics',
                                tracks_key=f'tracks_{tracking_filter}',
                                truths_key='truths'
                                )
                    for tracking_filter in tracking_filters]


    uncertainty_generators = [
        SumofCovarianceNormsMetric(generator_name=f'{tracking_filter} OSPA metrics',
                                tracks_key=f'tracks_{tracking_filter}')
        for tracking_filter in tracking_filters]

    associator = TrackToTruth(association_threshold=30)

    generators = ospa_generators + siap_generators + uncertainty_generators
    metric_manager = MultiManager(generators, associator=associator)

    metric_manager.add_data({'truths': [truth],
                            'tracks_unsmoothed': [track],
                            'tracks_CK-smoothed': [generated_states],
                            'tracks_RTS_smoothed': [KS_path],
                            })  
    metrics = metric_manager.generate_metrics()


    # sum up distance error from ground truth over all timestamps
    for tracking_filter in tracking_filters:
        total = sum([metrics[f'{tracking_filter} OSPA metrics']['OSPA distances'].value[i].value
                    for i in range(0, len(metrics[f'{tracking_filter} OSPA metrics']['OSPA distances'].value))])
        print(f'OSPA total value for {tracking_filter} is {total:.3f}')
        
    fig1 = MetricPlotter()
    fig1.plot_metrics(metrics, metric_names=['OSPA distances'])

In [ ]:
def plot_vel_example(truth,track,generated_states,KS_path):
    vel_plotter= Plotterly(dimension=Dimension.ONE)
    vel_plotter.plot_ground_truths(truth, [1],mode='lines', line=dict(width=1))
    vel_plotter.plot_tracks(track, [1], track_label='kalman filtered track')
    vel_plotter.plot_tracks(generated_states, [1], track_label='generated_states')
    vel_plotter.plot_tracks(KS_path, [1], track_label='Kalman-smoothed path')
    vel_plotter.fig.show()

# Step 1: Generate synthetic data

The state of the target can be represented as 2D Cartesian coordinates, $\left[x, \dot x, y, \dot y\right]^{\top}$, modelling both its position and velocity. A simple truth path is created with a sampling rate of 1 Hz.

In [ ]:
# np.random.seed(2024)

#THESE ARE the actual values for 'TAU'
q_x = 0.05
q_y = q_x
transition_model = CombinedLinearGaussianTransitionModel([ConstantVelocity(q_x),
                                                          ConstantVelocity(q_y)])

timesteps = [start_time]
truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])

num_steps = 50
for k in range(1, num_steps + 1):

    timesteps.append(start_time+timedelta(seconds=k))  # add next timestep to list of timesteps
    truth.append(GroundTruthState(
        transition_model.function(truth[k-1], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k]))


#THIS IS THE ACTUAL VALUE FOR 'SIGMA2'
noise_scale=0.5
measurement_model = LinearGaussian(
    ndim_state=4,  # Number of state dimensions (position and velocity in 2D)
    mapping=(0, 2),  # Mapping measurement vector index to state index
    noise_covar=np.array([[noise_scale, 0],  # Covariance matrix for Gaussian PDF
                          [0, noise_scale]])
    )

measurements = []
for state in truth:
    measurement = measurement_model.function(state, noise=True)
    measurements.append(Detection(measurement,
                                  timestamp=state.timestamp,
                                  measurement_model=measurement_model))


predictor = KalmanPredictor(transition_model)

updater = KalmanUpdater(measurement_model)

prior = GaussianState([[0], [1], [0], [1]], np.diag([1.5, 0.5, 1.5, 0.5]), timestamp=start_time)


time_interval=timedelta(seconds=1)


Our class:
0. A state generation function which takes given parameters and generates one ground truth based on input parameters, indicator variables. outputs Y0:n-1
1. CONDITONAL filtering algo (literally just a gaussian model but which takes a series of indicator variables, updating the model at each timestep to use different parameters based on 'K'=0 or 1, as well as the parameters for the model which aren't determined by K)
2. Smoothing algorithm (the above implemented thing, just with input series of indicator variables which tell us which model/matrix to use at each timestep)
3. A sample indicator variables function which takes states and parameters as input. 
4. An 'update parameters'(list[function(state_vectors, covariances)]) function which takes one function for each parameter, which itself is a function of the series of state_vectors and covariances at each timestep, and which outputs the new parameter variable
5. A sampler function which combines these, intiating using 0. then iterating through 1-4i where i indicates that we need to keep track of which parameter we're updating at each point. Should take num_iterations as a function and run it num_iterations*num_parameters time


#TODO: need to implement the recursive section:
    # basically find the probability of each parameter, e.g. sigma, given all the other parameters, e.g. tau!! 
    # it ISN'T a temporal shift of the same parameter, as we update this at each stage!! 
    # 1. so we do this initiation first, 
    # 3. Then then sample K over all the time states (RATHER than sampling K for single time point)
    # 4. Then update one parameter, 
    # 5. Then update the next parameter
    # 6. Then resample the states (repeating from state 1!)
    # THE ISSUE is that idk whether we resample the states after every parameter, or whether we update all states at once.

# Add model specifics

In [ ]:
sigma2=1
tau2=1
initial_parameters=[(sigma2, tau2)]

#Outline Transition model for K:

# Define the transition probabilities
p0 = 0.8  # Probability of staying in state 0
p1 = 0.7  # Probability of staying in state 1

# Define the transition matrix
transition_matrix = np.array([
    [p0, 1 - p1],  # Transition probabilities from state 0
    [1 - p0, p1]   # Transition probabilities from state 1
])

K_transition_matrix=transition_matrix
K_prior=CategoricalState(state_vector=StateVector([1,0]),timestamp=start_time)

parameter_posteriors= [sample_sigma2_posterior, sample_tau2_posterior]
X_prior=prior

CKsmoother=CarterKohnSampler(
    MCMCsample=True,
    measurements=measurements,
    parameters=initial_parameters,
    C=1, #indicator variables are irrelevant, both models are same
    X_prior=X_prior,
    K_prior=K_prior,
    K_transition_matrix=K_transition_matrix, #define model for K indicator states
    conditional_models_funct=conditional_models_funct, 
    parameter_posteriors=parameter_posteriors,
    )

generated_states, parameters, indicator_variables=CKsmoother.resample(num_iterations=1000)

In [ ]:
plot_sigma_tau(parameters, indicator_variables)
print(len(indicator_variables))

# Compare inferred trajectory with ground truth

In [ ]:
Kalman_smoother=KalmanSmoother(transition_model)
track = Track()
for measurement in measurements:
    prediction = predictor.predict(prior, timestamp=measurement.timestamp)
    hypothesis = SingleHypothesis(prediction, measurement)  # Group a prediction and measurement
    post = updater.update(hypothesis)
    track.append(post)
    prior = track[-1]

KS_path=Kalman_smoother.smooth(track)

plotter = Plotterly()
plotter.plot_ground_truths(truth, [0, 2])
plotter.plot_measurements(measurements, [0, 2])
plotter.plot_tracks(generated_states, [0, 2], uncertainty=True, track_label='generated_states')
plotter.plot_tracks(track, [0, 2], uncertainty=True)
plotter.plot_tracks(KS_path, [0, 2], uncertainty=True, track_label='Kalman-smoothed path')
plotter.fig

In [ ]:
plotOSPA_filtered_CK_RTS(truth,track,generated_states,KS_path)

In [ ]:
plot_vel_example(truth,track,generated_states,KS_path)

# Testing on Levy process

In [ ]:
# And the clock starts
start_time = datetime.now().replace(microsecond=0)

In [ ]:
seed = 1 # Random seem for reproducibility

# Driving process parameters
mu_W = 0.02
sigma_W2 = 4
alpha = 1.4
c=10
noise_case= NoiseCase.GAUSSIAN_APPROX

# Model parameters
theta=0.15

driver_x = AlphaStableNSMDriver(mu_W=mu_W, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, noise_case=noise_case)
driver_y = driver_x # Same driving process in both dimensions and sharing the same latents (jumps)
langevin_x = LevyLangevin(driver=driver_x, damping_coeff=theta, mu_W=mu_W)
langevin_y = LevyLangevin(driver=driver_y, damping_coeff=theta)
transition_model = CombinedLinearLevyTransitionModel([langevin_x, langevin_y])

timesteps = [start_time]

truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])

num_steps = 200

for k in range(num_steps):
    timesteps.append(start_time+timedelta(seconds=1*(k+1)))  # add next timestep to list of timesteps
    truth.append(GroundTruthState(
        transition_model.function(truth[k], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k+1]))

measurement_model = LinearGaussian(
    ndim_state=4,  # Number of state dimensions (position and velocity in 2D)
    mapping=(0, 2),  # Mapping measurement vector index to state index
    noise_covar=np.array([[1500, 0],  # Covariance matrix for Gaussian PDF
                          [0, 1500]])
    )

measurements = []
for state in truth:
    measurement = measurement_model.function(state, noise=True)
    measurements.append(Detection(measurement,
                                  timestamp=state.timestamp,
                                  measurement_model=measurement_model))


In [ ]:
sigma2=1
tau2=1
initial_parameters=[(sigma2, tau2)]

#Outline Transition model for K:

# Define the transition probabilities
p0 = 0.8  # Probability of staying in state 0
p1 = 0.7  # Probability of staying in state 1

# Define the transition matrix
transition_matrix = np.array([
    [p0, 1 - p1],  # Transition probabilities from state 0
    [1 - p0, p1]   # Transition probabilities from state 1
])

K_transition_matrix=transition_matrix
K_prior=CategoricalState(state_vector=StateVector([1,0]),timestamp=start_time)

parameter_posteriors= [sample_sigma2_posterior, sample_tau2_posterior]
X_prior=prior

CKsmoother=CarterKohnSampler(
    MCMCsample=True,
    measurements=measurements,
    parameters=initial_parameters,
    C=100, #second state has 100x the variance in transition model and measurement model
    X_prior=X_prior,
    K_prior=K_prior,
    K_transition_matrix=K_transition_matrix, #define model for K indicator states
    conditional_models_funct=conditional_models_funct, 
    parameter_posteriors=parameter_posteriors,
    )

generated_states, parameters, indicator_variables=CKsmoother.resample(num_iterations=200)

In [ ]:
print([int(np.argmax(state.state_vector)) for state in indicator_variables.states])

In [ ]:
plot_sigma_tau(parameters,indicator_variables)

In [ ]:
number_particles=5
predictor = MarginalisedParticlePredictor(transition_model=transition_model)
resampler = SystematicResampler()
updater = MarginalisedParticleUpdater(measurement_model, resampler)

# Sample from the prior Gaussian distribution
states = multivariate_normal.rvs(np.array([0, 1, 0, 1]),
                                  np.diag([1., 1., 1., 1.]),
                                  size=number_particles)
covars = np.stack([np.eye(4) * 100 for i in range(number_particles)], axis=2) # (M, M, N)

# Create prior particle state.
prior = MarginalisedParticleState(
    state_vector=StateVectors(states.T),
    covariance=covars,
    weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time-timedelta(seconds=1))

track = Track()
for measurement in measurements:
    prediction = predictor.predict(prior, timestamp=measurement.timestamp)
    hypothesis = SingleHypothesis(prediction, measurement)
    post = updater.update(hypothesis)
    track.append(post)
    prior = track[-1]
    
# Implement Particle Smoother Class
particlesmoother=ParticleSmoother(track=track)
conditionalsmoother=MarginalisedKalmanSmoother(track=track)
max_lag=None 
descendant_mean_track=particlesmoother= particlesmoother.particle_paths()
KS_path=conditionalsmoother.smooth(track=descendant_mean_track)

# Compare inferred trajectory with ground truth
plotter = Plotterly()
plotter.plot_ground_truths(truth, [0, 2])
plotter.plot_measurements(measurements, [0, 2])
plotter.plot_tracks(generated_states, [0, 2], track_label='generated_states')
plotter.plot_tracks(track, [0, 2], uncertainty=True)
plotter.plot_tracks(KS_path, [0, 2], uncertainty=True)

In [ ]:
plotOSPA_filtered_CK_RTS(truth,track,generated_states,KS_path)

In [ ]:
plot_vel_example(truth,track,generated_states,KS_path)